# 피처 선택 실험 — F1 개선 시도

`03_modeling.ipynb`의 XGBoost 베이스라인(전체 440개 피처, test F1 0.228)에서, 상관관계 필터 + 중요도 기반 피처 선택으로 성능이 개선되는지 확인한다.

**방법론 (test 누출 방지)**
1. `dev`(=train_raw+val, 원본 비율)에서 상관관계 필터(|corr|>0.95인 피처 쌍 중 하나 제거)로 중복 피처 축소
2. 남은 피처를 베이스라인 XGBoost 중요도로 정렬
3. 후보 K(피처 개수)마다 `03_modeling.ipynb`과 동일한 5-fold OOF 절차로 F1을 최대화하는 임계값 + OOF F1을 구함
4. **OOF F1이 가장 높은 K를 선택** — test는 아직 등장하지 않음
5. 선택된 K로 dev 전체를 SMOTE 재학습 후, test에 **최초 1회만** 적용해 baseline과 비교

In [1]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score, precision_recall_curve,
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

DATA_DIR = "../data/processed"
MODELS_DIR = "../models"
RANDOM_STATE = 42

XGB_PARAMS = dict(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
    random_state=RANDOM_STATE, n_jobs=-1,
)

## 1. 데이터 로드

In [2]:
X_train_raw = pd.read_csv(f"{DATA_DIR}/X_train_raw.csv")
y_train_raw = pd.read_csv(f"{DATA_DIR}/y_train_raw.csv").squeeze("columns")
X_val = pd.read_csv(f"{DATA_DIR}/X_val.csv")
y_val = pd.read_csv(f"{DATA_DIR}/y_val.csv").squeeze("columns")
X_test = pd.read_csv(f"{DATA_DIR}/X_test.csv")
y_test = pd.read_csv(f"{DATA_DIR}/y_test.csv").squeeze("columns")

X_dev = pd.concat([X_train_raw, X_val], ignore_index=True)
y_dev = pd.concat([y_train_raw, y_val], ignore_index=True)

print("X_dev:", X_dev.shape)
print("X_test:", X_test.shape)

X_dev: (1253, 440)
X_test: (314, 440)


## 2. 상관관계 필터

쌍별 절대 상관계수가 0.95를 넘는 피처 쌍 중 하나(뒤에 나오는 쪽)를 제거한다. 440개는 상관행렬(440x440)을 그대로 계산해도 충분히 빠르다.

In [3]:
corr_matrix = X_dev.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if (upper[col] > 0.95).any()]

corr_filtered_cols = [c for c in X_dev.columns if c not in to_drop]
print(f"상관관계 0.95 초과로 제거: {len(to_drop)}개 -> 남은 피처: {len(corr_filtered_cols)}개")

상관관계 0.95 초과로 제거: 173개 -> 남은 피처: 267개


## 3. 베이스라인 중요도로 피처 정렬

In [4]:
X_dev_res_base, y_dev_res_base = SMOTE(random_state=RANDOM_STATE).fit_resample(
    X_dev[corr_filtered_cols], y_dev
)
baseline_ranker = XGBClassifier(**XGB_PARAMS)
baseline_ranker.fit(X_dev_res_base, y_dev_res_base)

ranked_features = (
    pd.Series(baseline_ranker.feature_importances_, index=corr_filtered_cols)
    .sort_values(ascending=False)
    .index.tolist()
)
print("상위 10개:", ranked_features[:10])

상위 10개: ['feature_96', 'feature_487', 'feature_60', 'feature_29', 'feature_439', 'feature_113', 'feature_102', 'feature_156', 'feature_122', 'feature_337']


## 4. K별 5-fold OOF 평가

`03_modeling.ipynb`과 동일한 절차: 각 fold에서 SMOTE로 학습 후 나머지 1/5(원본 비율)에 예측, OOF 전체로 F1 최대화 임계값과 OOF F1을 구한다. K는 후보 몇 개만 비교한다 (전수 탐색이 아닌 탐색적 실험).

In [5]:
def run_oof(feature_subset, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof_proba = np.zeros(len(y_dev))
    X_sub = X_dev[feature_subset]

    for tr_idx, ho_idx in skf.split(X_sub, y_dev):
        X_tr, y_tr = X_sub.iloc[tr_idx], y_dev.iloc[tr_idx]
        X_ho = X_sub.iloc[ho_idx]
        X_tr_res, y_tr_res = SMOTE(random_state=RANDOM_STATE).fit_resample(X_tr, y_tr)
        fold_model = XGBClassifier(**XGB_PARAMS)
        fold_model.fit(X_tr_res, y_tr_res)
        oof_proba[ho_idx] = fold_model.predict_proba(X_ho)[:, 1]

    prec, rec, thr = precision_recall_curve(y_dev, oof_proba)
    f1s = 2 * prec * rec / (prec + rec + 1e-9)
    best_idx = f1s[:-1].argmax()
    return {
        "threshold": float(thr[best_idx]),
        "oof_f1": float(f1s[best_idx]),
        "oof_auroc": float(roc_auc_score(y_dev, oof_proba)),
    }

In [6]:
candidate_ks = [30, 50, 100, 200, len(corr_filtered_cols)]
grid_results = []

for k in candidate_ks:
    subset = ranked_features[:k]
    result = run_oof(subset)
    grid_results.append({"k": k, **result})
    print(f"k={k:>4} -> OOF F1={result['oof_f1']:.4f}, OOF AUROC={result['oof_auroc']:.4f}, threshold={result['threshold']:.4f}")

grid_df = pd.DataFrame(grid_results)
grid_df

k=  30 -> OOF F1=0.3039, OOF AUROC=0.7312, threshold=0.2445


k=  50 -> OOF F1=0.2819, OOF AUROC=0.7342, threshold=0.3308


k= 100 -> OOF F1=0.2659, OOF AUROC=0.7402, threshold=0.0616


k= 200 -> OOF F1=0.2717, OOF AUROC=0.7138, threshold=0.1731


k= 267 -> OOF F1=0.2534, OOF AUROC=0.7039, threshold=0.0669


,k,threshold,oof_f1,oof_auroc
0,30,0.244531,0.303922,0.731243
1,50,0.330794,0.281879,0.734198
2,100,0.061575,0.265928,0.740161
3,200,0.173054,0.271739,0.713788
4,267,0.066919,0.253425,0.703872


## 5. 최적 K 선택 및 test 최초 1회 평가

OOF F1이 가장 높은 K를 고르고, 그 K로 학습한 최종 모델을 test에 **한 번만** 적용한다. 03_modeling.ipynb 베이스라인(전체 440개 피처, test F1 0.228, AUROC 0.692)과 비교한다.

In [7]:
best_row = grid_df.loc[grid_df["oof_f1"].idxmax()]
best_k = int(best_row["k"])
best_threshold = float(best_row["threshold"])
best_features = ranked_features[:best_k]

print(f"선택된 K: {best_k} (OOF F1={best_row['oof_f1']:.4f})")

X_dev_res, y_dev_res = SMOTE(random_state=RANDOM_STATE).fit_resample(X_dev[best_features], y_dev)
final_model = XGBClassifier(**XGB_PARAMS)
final_model.fit(X_dev_res, y_dev_res)

test_proba = final_model.predict_proba(X_test[best_features])[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

comparison = pd.DataFrame([
    {
        "model": f"XGBoost + feature selection (k={best_k})",
        "precision": round(precision_score(y_test, test_pred, zero_division=0), 4),
        "recall": round(recall_score(y_test, test_pred, zero_division=0), 4),
        "f1": round(f1_score(y_test, test_pred, zero_division=0), 4),
        "auroc": round(roc_auc_score(y_test, test_proba), 4),
    },
    {
        "model": "XGBoost baseline (03_modeling.ipynb, k=440)",
        "precision": 0.1552, "recall": 0.4286, "f1": 0.2278, "auroc": 0.6919,
    },
])
comparison

선택된 K: 30 (OOF F1=0.3039)


,model,precision,recall,f1,auroc
0,XGBoost + feature selection (k=30),0.1290,0.1905,0.1538,0.7408
1,"XGBoost baseline (03_modeling.ipynb, k=440)",0.1552,0.4286,0.2278,0.6919


## 6. 결과 저장

In [8]:
comparison.to_csv(f"{MODELS_DIR}/feature_selection_comparison.csv", index=False)
grid_df.to_csv(f"{MODELS_DIR}/feature_selection_grid.csv", index=False)

improved = comparison.iloc[0]["f1"] > comparison.iloc[1]["f1"]
print("베이스라인 대비 F1 개선:", improved)
print(comparison)

베이스라인 대비 F1 개선: False
                                         model  precision  recall      f1  \
0           XGBoost + feature selection (k=30)     0.1290  0.1905  0.1538   
1  XGBoost baseline (03_modeling.ipynb, k=440)     0.1552  0.4286  0.2278   

    auroc  
0  0.7408  
1  0.6919  


## 결론

**베이스라인(k=440) 대비 개선되지 않음 — 프로덕션 모델은 그대로 유지한다.**

- OOF 기준으로는 k=30이 가장 좋았다(OOF F1 0.304, baseline보다 높음). 이 k로 학습해 test에 적용하면 AUROC는 0.692 -> 0.741로 뚜렷이 개선됐지만, F1은 0.228 -> 0.154로 오히려 떨어졌다(재현율 0.429 -> 0.190).
- 원인: dev의 양성 샘플이 83개뿐이라 5-fold OOF의 각 fold당 양성이 16~17개에 불과하고, 여기서 F1을 최대화하는 임계값을 고르면 fold별 노이즈에 민감해진다. 피처를 30개로 줄이면 모델이 더 날카롭게(AUROC 기준) 분리하지만, 그렇게 고른 임계값이 test의 실제 재현율 분포로는 잘 전이되지 않았다.
- 즉 **랭킹 품질(AUROC)은 개선됐지만 임계값 보정이 병목**이라는 뜻으로 읽힌다. 다음에 시도해볼 것: F1 argmax 대신 target-recall 기반 임계값 선택, 또는 임계값 자체를 여러 fold 평균으로 완만하게 추정.
- 그래서 `models/xgboost.joblib`(전체 440개 피처, test F1 0.228)을 프로덕션에 그대로 둔다. 이 실험은 `models/feature_selection_grid.csv` / `feature_selection_comparison.csv`로 결과만 남겨 참고한다.